# STAFF III — download, convert, label, visualize

[STAFF III](https://physionet.org/content/staffiii/1.0.0/) (104 patients, 1000 Hz).

Long 12-lead ECGs around balloon occlusion. Files are large, so **keep the local `data/staff_III` WFDB cache compressed** and only convert records you select.

For each selected recording:
1. Skip if already processed (`.npy` + `.pkl` + `.json`).
2. If `data/staff_III/data/{id}` exists, convert from there and **delete that cache copy**.
3. Otherwise download temporarily, convert, delete the temp WFDB.

In [1]:
from pathlib import Path
import sys

REPO = Path.cwd() if (Path.cwd() / "evaluation" / "common.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "evaluation"))

import pandas as pd
from IPython.display import display

import common as C

# --- subset ---
N_RECORDINGS = 8
RECORD_IDS = None  # e.g. ["039c", "077c"] to pin; None = filter below
PHASES = ["baseline_room", "baseline_cathlab", "inflation"]
ARTERIES = ["LAD", "RCA", "LCX"]
REQUIRE_EVENT_FILE = True
OVERWRITE_PROCESSED = False
CONSUME_WFDB = True
SEED = 42

RAW_DIR, PROC_DIR = C.dataset_dirs("staff_iii")
print("raw:", RAW_DIR)
print("processed:", PROC_DIR)
print("cache:", C.LOCAL_CACHES["staff_iii"])


raw: C:\Users\staso\OneDrive\Pulpit\Nauka\University\Thesis\code\ECG_Delineation\WTdelineator\data\evaluation\raw\staff_iii
processed: C:\Users\staso\OneDrive\Pulpit\Nauka\University\Thesis\code\ECG_Delineation\WTdelineator\data\evaluation\processed\staff_iii
cache: C:\Users\staso\OneDrive\Pulpit\Nauka\University\Thesis\code\ECG_Delineation\WTdelineator\data\staff_III


## 1. Official annotation spreadsheet

In [2]:
ann = C.load_staff_catalogue()
print(ann.groupby(["phase", "occluded_artery"], dropna=False).size().unstack(fill_value=0))
display(ann.head())


occluded_artery   LAD  LCX  LM  RCA  NaN
phase                                   
baseline_cathlab    0    0   0    0  114
baseline_room       0    0   0    0   73
inflation          58   33   3   58    0
post_cathlab        0    0   0    0   95
post_room           0    0   0    0   96


,patient_id,age,sex,record_id,phase,phase_slot,occluded_artery_raw,occluded_artery,d0_s,d1_s,d2_s,injection_times_s,location_flag
0,1,None,None,001a,baseline_room,BR,None,None,NaN,NaN,NaN,[],None
1,1,None,None,001b,baseline_cathlab,BC1,None,None,NaN,NaN,NaN,[],None
2,1,None,None,001d,post_room,PR1,None,None,NaN,NaN,NaN,[],None
3,1,None,None,001c,inflation,BI1,mid RCA,RCA,60.0,185.0,0.0,[],None
4,2,None,None,002a,baseline_room,BR,None,None,NaN,NaN,NaN,[],None


In [17]:
ann.occluded_artery.value_counts(), ann.occluded_artery.isna().sum()

(occluded_artery
 RCA    58
 LAD    58
 LCX    33
 LM      3
 Name: count, dtype: int64,
 np.int64(378))

In [ ]:
ann[ann.patient_id==39]

,patient_id,age,sex,record_id,phase,phase_slot,occluded_artery_raw,occluded_artery,d0_s,d1_s,d2_s,injection_times_s,location_flag
192,39,None,None,039a,baseline_room,BR,None,None,NaN,NaN,NaN,[],None
193,39,None,None,039b,baseline_cathlab,BC1,None,None,NaN,NaN,NaN,[],None
194,39,None,None,039e,post_room,PR1,None,None,NaN,NaN,NaN,[],None
195,39,None,None,039c,inflation,BI1,mid LAD,LAD,0.0,285.0,7.0,[],None
196,39,None,None,039c,inflation,BI2,prox LAD,LAD,292.0,200.0,30.0,[],None
197,39,None,None,039c,inflation,BI3,prox LAD,LAD,522.0,137.0,106.0,"[502.0, 707.0]",None
198,39,None,None,039d,inflation,BI4,prox LAD,LAD,0.0,152.0,60.0,[],None
199,39,None,None,039d,inflation,BI5,prox LAD,LAD,212.0,117.0,36.0,[],None


## 2. Choose a subset

In [22]:
import numpy as np

if RECORD_IDS:
    record_ids = [C.staff_record_id(x) for x in RECORD_IDS]
    selected = ann[ann["record_id"].isin(record_ids)].drop_duplicates("record_id")
else:
    pool = ann.copy()
    if PHASES:
        pool = pool[pool["phase"].isin(PHASES)]
    if ARTERIES:
        keep_phase = pool["phase"] != "inflation"
        keep_artery = pool["occluded_artery"].isin(ARTERIES)
        pool = pool[keep_phase | keep_artery]
    pool = pool.drop_duplicates(subset=["record_id", "phase_slot"])

    rng = np.random.default_rng(SEED)
    infl = pool[pool["phase"] == "inflation"].copy()
    if REQUIRE_EVENT_FILE:
        infl = infl[infl["d0_s"].notna() | infl["d1_s"].notna()]

    picked_ids = []
    if ARTERIES and len(infl):
        for artery in ARTERIES:
            cand = infl[infl["occluded_artery"] == artery]
            if len(cand):
                picked_ids.append(cand.sample(n=1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]["record_id"])

    patients = ann.loc[ann["record_id"].isin(picked_ids), "patient_id"].dropna().astype(int).unique().tolist() if picked_ids else []
    extra = pool[pool["patient_id"].isin(patients)] if patients else pool
    selected = extra.drop_duplicates("record_id")
    if len(selected) > N_RECORDINGS:
        infl_sel = selected[selected["phase"] == "inflation"]
        rest = selected[selected["phase"] != "inflation"]
        n_rest = max(0, N_RECORDINGS - len(infl_sel))
        selected = pd.concat([infl_sel, rest.head(n_rest)]).drop_duplicates("record_id").head(N_RECORDINGS)
    record_ids = selected["record_id"].tolist()

todo = C.unprocessed_ids(PROC_DIR, record_ids, overwrite=OVERWRITE_PROCESSED)
print(f"Selected {len(record_ids)}; already processed {len(record_ids) - len(todo)}; to acquire {len(todo)}")
display(selected[["patient_id", "record_id", "phase", "occluded_artery", "d0_s", "d1_s", "d2_s"]])


Selected 8; already processed 8; to acquire 0


,patient_id,record_id,phase,occluded_artery,d0_s,d1_s,d2_s
195,39,039c,inflation,LAD,0.0,285.0,7.0
198,39,039d,inflation,LAD,0.0,152.0,60.0
382,77,077c,inflation,RCA,0.0,123.0,62.0
474,96,096c,inflation,LCX,94.0,299.0,347.0
475,96,096d,inflation,LCX,9.0,296.0,64.0
476,96,096e,inflation,LCX,0.0,164.0,136.0
192,39,039a,baseline_room,None,NaN,NaN,NaN
193,39,039b,baseline_cathlab,None,NaN,NaN,NaN


## 3. Convert from cache or temp download

In [ ]:
results = [
    C.acquire_and_convert_staff(rid, ann, overwrite=OVERWRITE_PROCESSED, consume=CONSUME_WFDB)
    for rid in record_ids
]
print(C.summarize_acquire(results))
for row in results:
    extra = f" origin={row.get('origin')}" if row.get("origin") else ""
    print(f"  {row['record_id']}: {row['status']}{extra}")
failed = [r for r in results if r["status"] == "failed"]
if failed:
    print("failed:", failed)

index = C.load_index(PROC_DIR)
display(index)


## 4. Visualize

In [34]:
index = C.load_index(PROC_DIR)
infl_rows = index[index["phase"] == "inflation"]
example_id = str((infl_rows if len(infl_rows) else index).iloc[0]["record_id"])
example_id = "039d"
print("plotting", example_id)
_ = C.plot_staff_record(example_id, PROC_DIR)


plotting 039d


In [24]:
index = C.load_index(PROC_DIR)
_ = C.plot_category_counts(index["phase"].value_counts(), title="STAFF III subset — recordings by phase", color="#e6550d")
artery_counts = index["occluded_artery"].fillna("none (baseline/post)").value_counts()
_ = C.plot_category_counts(artery_counts, title="STAFF III subset — occluded artery", color="#de2d26")
